In [ ]:
"""Q1 - 학생 성적 보고서 생성기"""
import csv, json, logging

logging.basicConfig(level=logging.INFO, 
                    format="[%(levelname)s] %(message)s")

def make_report(csv_path: str, json_path: str) -> int:
    students = []
    success = 0
    try: 
        with open (csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                scores_values = [row["중간"], row["기말"], row["과제"]]
                if "" in scores_values:
                    mid = None if row["중간"] == "" else int(row["중간"])
                    fin = None if row["기말"] == "" else int(row["기말"])
                    hw = None if row["과제"] == "" else int(row["과제"])
                    student_scores = {"중간": mid, "기말": fin, "과제": hw}
                    average = None
                    grade = None
                   
                else:
                    mid = int(row["중간"])
                    fin = int(row["기말"])
                    hw = int(row["과제"])

                    student_scores = {"중간": mid, "기말": fin, "과제": hw}
                    average = ((mid*0.3)+(fin*0.5)+(hw*0.2))                    
                    if average>=90:
                        grade = "A"
                    elif 80<=average<90:
                        grade = "B"
                    elif 70<=average<80:
                        grade = "C"
                    else:
                        grade = "F"

                student = {
                    "이름": row["이름"], 
                    "학번": row["학번"], 
                    "점수": student_scores, 
                    "평균": average, 
                    "등급": grade
                    }
                students.append(student)

                if average == None:
                    logging.info(f"{student['이름']}: 점수가 모두 채워지지 않았습니다.")
                else:
                    logging.info(f"{student['이름']}: {student['평균']}, {student['등급']}")
                    success += 1

        with open (json_path, "w", encoding="utf-8") as f:
             json.dump(students, f,
                       ensure_ascii=False,
                       indent=2)
        return success
           
    except FileNotFoundError:
        logging.warning(f"파일이 존재하지 않습니다.: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error("인코딩이 잘못되었습니다.")
        return 0

    
make_report("scores.csv", "report.json")

[INFO] 김언어: 89.5, B
[INFO] 이국문: 84.4, B
[INFO] 박영문: 93.5, A
[INFO] 최역사: 점수가 모두 채워지지 않았습니다.


3

<report.json 내용>
[
  {
    "이름": "김언어",
    "학번": "2026-10000",
    "점수": {
      "중간": 85,
      "기말": 92,
      "과제": 90
    },
    "평균": 89.5,
    "등급": "B"
  },
  {
    "이름": "이국문",
    "학번": "2026-12345",
    "점수": {
      "중간": 78,
      "기말": 88,
      "과제": 85
    },
    "평균": 84.4,
    "등급": "B"
  },
  {
    "이름": "박영문",
    "학번": "2026-13579",
    "점수": {
      "중간": 95,
      "기말": 90,
      "과제": 100
    },
    "평균": 93.5,
    "등급": "A"
  },
  {
    "이름": "최역사",
    "학번": "2025-11111",
    "점수": {
      "중간": null,
      "기말": 82,
      "과제": 88
    },
    "평균": null,
    "등급": null
  }
]

설명.
with문으로 파일을 열어 예외 상황에도 파일이 잘 닫히도록 했고, 한글 텍스트의 표준인 utf-8 인코딩을 지정해 인코딩 오류가 발생할 확률을 줄이고자 했다. 결측값은 점수만 모인 리스트를 따로 만들어 빈 문자열이 있는지 확인하고, 빈칸인 과목과 평균, 등급을 모두 None으로 변경해 JSON파일에서도 null로 변환되도록 했다. 예외 처리는 FileNotFoundError와 UnicodeDecodeError로 except에 명시했고 각각 0을 반환하도록 만들었으며, 최종 JSON 파일을 저장할 때 ensure_ascii = False, indent=2 형식을 지정해 한글이 보존되고 들여쓰기로 읽기 쉽게 했다.

In [44]:
"""Q2. 사용자 정의 예외와 자모 분류"""

class InvalidJamoError (ValueError):
    """한글 자모 이외의 한 글자 문자일 때."""

def classify_jamo(c:str) -> str:
    if not isinstance(c, str): 
        raise TypeError(f"문자열 형태가 아닙니다.")
    elif len(c) != 1:
        raise ValueError(f"한 글자가 아닙니다.")
    elif 0x3131 <= ord(c) <= 0x314e:
        return "자음"
    elif 0x314f <= ord(c) <= 0x3163:
        return "모음"
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다. : {c}")

inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]
for i in inputs:
    try:
        print(classify_jamo(i))
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except TypeError as e:
        print(f"[TypeError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")


자음
모음
자음
[InvalidJamoError] 한글 자모가 아닙니다. : 가
[ValueError] 한 글자가 아닙니다.
[TypeError] 문자열 형태가 아닙니다.
자음
모음
[ValueError] 한 글자가 아닙니다.


설명.
ValueError는 파이썬에서 자료형의 종류는 맞지만 내용이 틀렸다는 뜻의 예외이고, InvalidJamoError도 자료형은 맞지만 값이 한글 자모가 아닐 때 발생하므로 포괄적인 Exception보다는 ValueError의 자식 예외로 정의하는 것이 적절하다. 함수에서는 타입과 길이를 먼저 검사한 후, 유니코드 범위로 입력값이 한글 자모가 맞는지 확인했으며 예외는 모두 InvalidJamoError를 발생시키도록 했다. 주어진 입력값은 리스트이기 때문에 함수를 실행할 때 for문을 사용해 각 원소를 순회했고, 예외의 경우 except로 구분하고 e로 받아 프로그램이 멈추지 않고 예외의 종류와 메시지를 출력하도록 했다.